# STG-NF Pose Ablation: AlphaPose vs YOLO-pose

Runs the **same** STG-NF checkpoint on two different pose extractions of the **same** clip and
reports the standalone frame-level Micro AUC of each, so you can quantify the gain from swapping
AlphaPose for YOLO-pose.

* AlphaPose pipeline: YOLOX-X detector + ResNet-152 pose + PoseFlow/ReID tracking (already cached from `PRISM_Test` Step 6c).
* YOLO-pose pipeline: Ultralytics YOLO26-pose (detection + pose in one pass) + greedy IoU tracking (extracted here).

The two poses are fed through **identical** STG-NF inference code, so the only variable is the pose source.



## Step 1: Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



## Step 2: Install Ultralytics YOLO26-pose

Pure-Python install (no CUDA compilation); `yolo26l-pose.pt` auto-downloads on first use.
Run once per session (or skip if already installed).



In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install ultralytics

import ultralytics
print("ultralytics", ultralytics.__version__)


## Step 3: Configuration

Paths match `PRISM_Test` defaults. Change only what you need.



In [ ]:
from pathlib import Path

# Same video / image-frames directory used by PRISM_Test.
VIDEO_PATH = "/content/drive/MyDrive/Experiments/Videos/1"
VIDEO_STEM = Path(VIDEO_PATH).stem

# STG-NF checkpoint + training-time hyper-parameters (must match the checkpoint).
STGNF_CHECKPOINT   = "/content/drive/MyDrive/STG-NF/original_shanghaitech/logs/ShanghaiTech/ShanghaiTech_84/ShanghaiTech_84.tar"
STGNF_DATASET      = "Avenue"
STGNF_SEG_LEN      = 24
STGNF_SMOOTH_SIGMA = 7.0
STGNF_ATTENTION    = "none"

# Manual ground-truth labels (1-D 0/1 array, 1 = abnormal).
GROUND_TRUTH_DIR       = Path("/content/drive/MyDrive/Experiments/ground_truth_labels")
GROUND_TRUTH_LABEL_FILE = "clip_4_labels.npy"   # set None to auto-resolve

# Where AlphaPose already cached its tracked-person JSON (from PRISM_Test Step 6c).
ALPHAPOSE_CACHE = Path("/content/drive/MyDrive/Experiments/custom_video_cache/stgnf_pose")

# Outputs
OUTPUT_ROOT = Path("/content/drive/MyDrive/Experiments") / VIDEO_STEM / "pose_ablation"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# YOLO-pose extraction knobs
DET_THR  = 0.3     # person detection confidence threshold
YOLO_MODEL = "yolo26l-pose.pt"   # Ultralytics YOLO26-pose (detection + pose in one pass)
TRACKER  = "bytetrack.yaml"   # Ultralytics built-in tracker: "bytetrack.yaml" or "botsort.yaml"

print("video stem :", VIDEO_STEM)
print("checkpoint :", STGNF_CHECKPOINT)
print("output     :", OUTPUT_ROOT)



## Step 4: Clone STG-NF + define the YOLO-pose->tracked-JSON converter

STG-NF's `dataset.py` parses `scene_id, clip_id = filename.split('_')[:2]` and `single_pose_dict2np`
reads `{pid: {frame_key: {keypoints, scores}}}` with COCO-17 keypoints. YOLO-pose already emits COCO-17
in the same order, so we only convert the schema (no reordering — `keypoints17_to_coco18` does that).



In [ ]:
import os, sys, subprocess, json
from pathlib import Path
import numpy as np

STGNF_REPO = "/content/STG-NF"
if not Path(STGNF_REPO).exists():
    subprocess.run(["git", "clone", "https://github.com/Hadi6618/STG-NF.git", STGNF_REPO], check=True)
print("STG-NF repo:", STGNF_REPO)


def yolopose_to_tracked_person(keypoints, keypoint_scores, det_scores, track_ids, frame_indices, num_digits=4):
    # [N,17,2] + [N,17] + [N] + [N] + [N] -> {pid: {frame_key: {keypoints: [x,y,c]*17, scores: scalar}}}
    # `scores` must be a single scalar (detection confidence): STG-NF stacks it
    # per frame and reshapes to (1, seg_len). Per-keypoint confidence stays in the [x,y,c] triples.
    tracked = {}
    for i in range(len(track_ids)):
        pid = str(int(track_ids[i]))
        fk = str(int(frame_indices[i])).zfill(num_digits)
        kp_flat = []
        for (x, y), c in zip(keypoints[i], keypoint_scores[i]):
            kp_flat.extend([float(x), float(y), float(c)])
        tracked.setdefault(pid, {})[fk] = {
            "keypoints": kp_flat,
            "scores": float(det_scores[i]),
        }
    return tracked


def write_tracked_person(tracked, path):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(tracked, f)
    return p


print("converter ready")



## Step 5: Extract poses with YOLO26-pose + ByteTrack tracking

Produces `01_0001_yolopose_tracked_person.json` (same schema as AlphaPose). ByteTrack
(motion + Kalman association) keeps person IDs stable through crossings and brief
occlusions, which STG-NF relies on since it models each person's pose sequence independently.
Set `TRACKER = "botsort.yaml"` in the config cell to add appearance (ReID) features.



In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import torch
import cv2

from ultralytics import YOLO

device = "cuda:0" if torch.cuda.is_available() else "cpu"

pose_model = YOLO(YOLO_MODEL)
print("pose model:", YOLO_MODEL)


def iter_frames(video_path):
    if os.path.isdir(video_path):
        exts = (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp")
        files = sorted(f for f in os.listdir(video_path) if f.lower().endswith(exts))
        for i, f in enumerate(files):
            img = cv2.imread(os.path.join(video_path, f))
            if img is not None:
                yield i, cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        cap = cv2.VideoCapture(video_path)
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            yield i, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            i += 1
        cap.release()


# ByteTrack (Ultralytics built-in) assigns stable track IDs via motion + Kalman.
# `persist=True` keeps tracker state across the frames yielded by iter_frames().

all_kps, all_scores, all_det_scores, all_tids, all_frames = [], [], [], [], []
for frame_idx, frame_rgb in iter_frames(VIDEO_PATH):
    res = pose_model.track(frame_rgb, verbose=False, persist=True, tracker=TRACKER)[0]
    if res.boxes is None or len(res.boxes) == 0 or res.keypoints is None:
        continue
    confs = res.boxes.conf.cpu().numpy()

    keep = confs > DET_THR   # YOLO-pose is person-only, so confidence is the only filter
    if not keep.any():
        continue
    confs = confs[keep]                          # (M,) person-detection confidence
    kp = res.keypoints.xy.cpu().numpy()[keep]    # (M, 17, 2)
    kc = res.keypoints.conf.cpu().numpy()[keep]  # (M, 17)

    if res.boxes.id is not None:
        tids = res.boxes.id.cpu().numpy()[keep].astype(np.int64)
    else:
        tids = np.arange(len(confs), dtype=np.int64)  # fallback (rare): tracking unavailable

    for i in range(len(confs)):
        all_kps.append(kp[i])
        all_scores.append(kc[i])
        all_det_scores.append(float(confs[i]))
        all_tids.append(int(tids[i]))
        all_frames.append(frame_idx)

    if frame_idx % 100 == 0:
        print(f"frame {frame_idx}: {len(confs)} person(s)")

all_kps        = np.stack(all_kps) if all_kps else np.zeros((0, 17, 2), dtype=np.float32)
all_scores     = np.stack(all_scores) if all_scores else np.zeros((0, 17), dtype=np.float32)
all_det_scores = np.asarray(all_det_scores, dtype=np.float64)
all_tids       = np.asarray(all_tids, dtype=np.int64)
all_frames = np.asarray(all_frames, dtype=np.int64)
print(f"instances: {len(all_tids)} across {len(np.unique(all_frames))} frames")

tracked = yolopose_to_tracked_person(all_kps, all_scores, all_det_scores, all_tids, all_frames, num_digits=4)

local_dir = Path("/content/yolopose_pose_work") / VIDEO_STEM
json_path = write_tracked_person(tracked, local_dir / "01_0001_yolopose_tracked_person.json")

drive_dir = OUTPUT_ROOT / "yolopose_pose"
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(json_path, drive_dir / json_path.name)
print("saved :", json_path)
print("cached:", drive_dir / json_path.name)



## Step 6: STG-NF scoring helper (identical code path for both pose sources)

This is the same per-frame scoring logic as `PRISM_Test` Step 6d, refactored into a function so the
AlphaPose and YOLO-pose poses are scored by **exactly** the same code.



In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import cv2
from scipy.ndimage import gaussian_filter1d


def score_stgnf(pose_dir, video_path, checkpoint, dataset_name, seg_len, smooth_sigma,
                attention, device, stgnf_repo="/content/STG-NF"):
    # Return (raw_log_likelihood, smoothed_log_likelihood, num_frames, fps).
    os.chdir(stgnf_repo)
    if stgnf_repo not in sys.path:
        sys.path.insert(0, stgnf_repo)
    for name in list(sys.modules):
        if name in ("dataset", "args", "gen_data") or name.startswith("utils") or name.startswith("models"):
            del sys.modules[name]

    from args import init_parser, init_sub_args
    from dataset import get_dataset_and_loader
    from models.STG_NF.model_pose import STG_NF
    from models.training import Trainer
    from utils.data_utils import trans_list
    from utils.optim_init import init_optimizer, init_scheduler
    from utils.train_utils import init_model_params

    if os.path.isdir(video_path):
        exts = (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp")
        num_frames = len([f for f in os.listdir(video_path) if f.lower().endswith(exts)])
        fps = 25.0
        print(f"Image dir: {num_frames} frames @ {fps:.2f} FPS")
    else:
        cap = cv2.VideoCapture(video_path)
        num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        if not fps or fps <= 0:
            fps = 25.0
        print(f"Video: {num_frames} frames @ {fps:.2f} FPS")

    argv = [
        "--dataset", dataset_name,
        "--pose_path_test", pose_dir,
        "--pose_path_train", pose_dir,
        "--vid_path_test", str(Path(video_path).parent),
        "--vid_path_train", str(Path(video_path).parent),
        "--checkpoint", checkpoint,
        "--seg_len", str(seg_len),
        "--batch_size", "256",
        "--num_workers", "2",
        "--device", device,
    ]
    if attention != "none":
        argv += ["--attention", attention]

    args = init_parser().parse_args(argv)
    args, _ = init_sub_args(args)

    dataset, loader = get_dataset_and_loader(args, trans_list=trans_list, only_test=True)
    model_args = init_model_params(args, dataset)
    model = STG_NF(**model_args)
    trainer = Trainer(
        args, model, loader["train"], loader["test"],
        optimizer_f=init_optimizer(args.model_optimizer, lr=args.model_lr),
        scheduler_f=init_scheduler(args.model_sched, lr=args.model_lr, epochs=args.epochs),
    )
    trainer.load_checkpoint(checkpoint)
    normality_scores = trainer.test()
    metadata = dataset["test"].metadata

    meta_np = np.array(metadata)
    if meta_np.size == 0:
        raise ValueError("No pose segments detected for this pose source.")
    person_ids = set(meta_np[:, 2].tolist())
    per_person = {pid: np.full(num_frames, np.inf, dtype=np.float64) for pid in person_ids}
    for pid in person_ids:
        inds = np.where(meta_np[:, 2] == pid)[0]
        frame_inds = meta_np[inds, 3].astype(int) + seg_len // 2
        for score, fidx in zip(normality_scores[inds], frame_inds):
            if 0 <= fidx < num_frames:
                per_person[pid][fidx] = score
    clip_score = np.amin(np.stack(list(per_person.values())), axis=0)
    finite = np.isfinite(clip_score)
    if finite.any():
        clip_score[~finite] = clip_score[finite].max()
    else:
        clip_score[:] = 0.0

    raw_ll = clip_score.astype(np.float32)
    smoothed_ll = gaussian_filter1d(raw_ll, sigma=smooth_sigma).astype(np.float32)
    return raw_ll, smoothed_ll, num_frames, fps


print("score_stgnf ready")



## Step 7: Score both pose sources with the same checkpoint


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

# --- AlphaPose: copy the cached tracked JSON into its own dir ---
alpha_dir = Path("/content/stgnf_pose_work") / f"{VIDEO_STEM}_alphapose"
alpha_dir.mkdir(parents=True, exist_ok=True)
alpha_src = ALPHAPOSE_CACHE / "01_0001_alphapose_tracked_person.json"
if not alpha_src.exists():
    hits = list(ALPHAPOSE_CACHE.rglob("*_alphapose_tracked_person.json"))
    if not hits:
        raise FileNotFoundError(
            f"No AlphaPose tracked JSON under {ALPHAPOSE_CACHE}. Run PRISM_Test Step 6c first."
        )
    alpha_src = hits[0]
shutil.copy2(alpha_src, alpha_dir / "01_0001_alphapose_tracked_person.json")
print("AlphaPose pose:", alpha_src)

# --- YOLO-pose: written by Step 5 ---
rtm_dir = Path("/content/yolopose_pose_work") / VIDEO_STEM

# --- Score both ---
alpha_raw, alpha_smooth, n_a, fps_a = score_stgnf(
    str(alpha_dir), VIDEO_PATH, STGNF_CHECKPOINT, STGNF_DATASET,
    STGNF_SEG_LEN, STGNF_SMOOTH_SIGMA, STGNF_ATTENTION, device,
)
print(f"AlphaPose scoring done: {n_a} frames @ {fps_a:.2f} FPS")

rtm_raw, rtm_smooth, n_r, fps_r = score_stgnf(
    str(rtm_dir), VIDEO_PATH, STGNF_CHECKPOINT, STGNF_DATASET,
    STGNF_SEG_LEN, STGNF_SMOOTH_SIGMA, STGNF_ATTENTION, device,
)
print(f"YOLO-pose scoring done: {n_r} frames @ {fps_r:.2f} FPS")

np.save(OUTPUT_ROOT / "alphapose_smooth_ll.npy", alpha_smooth)
np.save(OUTPUT_ROOT / "yolopose_smooth_ll.npy", rtm_smooth)
print("saved smoothed log-likelihoods to", OUTPUT_ROOT)



## Step 8: Standalone AUC comparison + overlaid curves


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def oriented_auc(scores, y):
    if len(np.unique(y)) < 2:
        return None, "undefined"
    a_pos = float(roc_auc_score(y, scores))
    a_neg = float(roc_auc_score(y, -scores))
    if a_neg > a_pos:
        return a_neg, "normality->flipped"
    return a_pos, "anomaly"


# Ground truth
if GROUND_TRUTH_LABEL_FILE:
    gt_path = GROUND_TRUTH_DIR / GROUND_TRUTH_LABEL_FILE
else:
    gt_path = GROUND_TRUTH_DIR / f"{VIDEO_STEM}_labels.npy"
    if not gt_path.exists():
        candidates = sorted(GROUND_TRUTH_DIR.glob("*.npy"))
        gt_path = candidates[0] if len(candidates) == 1 else (GROUND_TRUTH_DIR / "clip_1_labels.npy")
gt = (np.asarray(np.load(gt_path, allow_pickle=True), dtype=np.float64).reshape(-1) > 0).astype(np.uint8)

# Both scores are *normality* (log-likelihood); negate to get anomaly score.
n = min(len(gt), len(alpha_smooth), len(rtm_smooth))
Y = gt[:n]
A_anom = (-alpha_smooth[:n]).astype(np.float64)
R_anom = (-rtm_smooth[:n]).astype(np.float64)

auc_alpha, mode_a = oriented_auc(A_anom, Y)
auc_rtm,   mode_r = oriented_auc(R_anom, Y)

print("=" * 64)
print(f"GT: {gt_path.name}  frames={n}  normal={(Y == 0).sum()}  abnormal={(Y == 1).sum()}")
print(f"STG-NF + AlphaPose  : AUC = {auc_alpha * 100:.4f}%   (polarity: {mode_a})")
print(f"STG-NF + YOLO-pose    : AUC = {auc_rtm * 100:.4f}%   (polarity: {mode_r})")
print(f"Delta (YOLO-pose - AlphaPose): {(auc_rtm - auc_alpha) * 100:+.4f} pts")
print("=" * 64)

# per-frame CSV + JSON report
frame_csv = OUTPUT_ROOT / "stgnf_pose_ablation_scores.csv"
pd.DataFrame({
    "frame_index": np.arange(n),
    "gt": Y,
    "stgnf_alphapose_anomaly": A_anom,
    "stgnf_yolopose_anomaly": R_anom,
}).to_csv(frame_csv, index=False)

report = {
    "video": VIDEO_STEM,
    "label_file": str(gt_path),
    "aligned_frames": int(n),
    "alphapose_auc": float(auc_alpha),
    "yolopose_auc": float(auc_rtm),
    "delta_pts": float(auc_rtm - auc_alpha),
}
(OUTPUT_ROOT / "stgnf_pose_ablation_report.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

# overlaid curves (downsampled for readability)
step = max(1, n // 2000)
xs = np.arange(n)[::step]
plt.figure(figsize=(14, 5))
plt.plot(xs, A_anom[::step], label=f"AlphaPose  (AUC {auc_alpha * 100:.2f}%)", alpha=0.85, linewidth=1.0)
plt.plot(xs, R_anom[::step], label=f"YOLO-pose    (AUC {auc_rtm * 100:.2f}%)", alpha=0.85, linewidth=1.0)
plt.fill_between(xs, 0, 1, where=(Y[::step] == 1), color="red", alpha=0.15, label="GT anomaly")
plt.xlabel("frame")
plt.ylabel("anomaly score")
plt.title(f"STG-NF anomaly score: AlphaPose vs YOLO-pose ({VIDEO_STEM})")
plt.legend(loc="upper right")
plot_path = OUTPUT_ROOT / "stgnf_pose_ablation_curves.png"
plt.tight_layout()
plt.savefig(plot_path, dpi=120)

print("saved:", frame_csv)
print("saved:", plot_path)
print("saved:", OUTPUT_ROOT / "stgnf_pose_ablation_report.json")

